In [1]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
import numpy as np
import os
from io import BytesIO

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
container_name = "processeddata"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

In [2]:
def load_blob_csv(blob_name):
    blob_client = blob_service_client.get_blob_client(
        container=container_name,
        blob=blob_name
    )

    data = blob_client.download_blob().readall()
    return pd.read_csv(BytesIO(data))

# EKSEMPEL (tilpass til dine faktiske filer)
df_consumption = load_blob_csv("FRIKSTAD_processed.csv")
df_weather = load_blob_csv("FRIKSTAD_weather.csv")
df_norgespris = load_blob_csv("frikstad_norgespris.csv")

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../src")))

from analysis.regression import (
    prepare_norgespris_regression_data
)

# -------------------------------------------------------
# 1. DATA PREP (samme som før)
# -------------------------------------------------------

regression_df = prepare_norgespris_regression_data(
    df_consumption,
    df_weather,
    df_norgespris,
    station_customers_total=None,
    exclude_consumption_codes=(26,),
)

df = regression_df.copy()

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["log_value_kwh"] = np.log1p(df["value_kwh"])

# -------------------------------------------------------
# 2. PANEL INDEX (VIKTIG)
# -------------------------------------------------------

panel_df = df.set_index(
    ["metering_point_anonymous", "timestamp"]
).sort_index()

# -------------------------------------------------------
# 3. REDUSER MEMORY (viktig for 20M rader)
# -------------------------------------------------------

cols_needed = [
    "log_value_kwh",
    "norgespris_share",
    "air_temperature",
    "wind_speed",
    "precipitation_mm",
    "is_weekend",
    "is_holiday",
    "hour",
    "month"
]

panel_df = panel_df[cols_needed].copy()

# -------------------------------------------------------
# 4. FIX CATEGORICALS (hindrer eksplosjon)
# -------------------------------------------------------

panel_df["hour"] = panel_df["hour"].astype("category")
panel_df["month"] = panel_df["month"].astype("category")

# -------------------------------------------------------
# 5. MODELL (HER ER FIXEN)
# -------------------------------------------------------

y = panel_df["log_value_kwh"]

X = panel_df[[
    "norgespris_share",
    "air_temperature",
    "wind_speed",
    "precipitation_mm",
    "is_weekend",
    "is_holiday",
    "hour",
    "month"
]]

model = PanelOLS(
    y,
    X,
    entity_effects=True,   # fixed effects (kundenivå)
    drop_absorbed=True     # VIKTIG: unngår FE + collinearity crash
)

# -------------------------------------------------------
# 6. FIT (skalert for store datasett)
# -------------------------------------------------------

results = model.fit(
    cov_type="clustered",
    cluster_entity=True,
    low_memory=True,
    use_lsmr=True
)

# -------------------------------------------------------
# 7. METRICS (samme som før)
# -------------------------------------------------------

beta_share = results.params["norgespris_share"]

metrics = {
    "nobs": results.nobs,
    "r2_within": results.rsquared_within,
    "beta_share": beta_share,
    "effect_10pp_pct": (np.exp(beta_share * 0.10) - 1) * 100,
    "effect_full_share_pct": (np.exp(beta_share) - 1) * 100,
}

print(results.summary)
print(metrics)

/opt/anaconda3/envs/bachelor2026/lib/python3.11/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
